In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
import os 
os.getcwd()

In [ ]:
cd donnees_personnelles

In [ ]:
import re
import unicodedata
import pandas as pd
from datetime import datetime

# --- helper functions ---
def norm_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip().lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def split_rules(s):
    if not s:
        return []
    return [p.strip() for p in re.split(r"(?i)\bET\b[,;:]?\s*", s) if p.strip()]

cols_to_keep=["Libellé du Domaine","Libellé du Sous-domaine","Libellé de l'évènement","Libellé Population","Codification Population","Général / Particulier","Passant / Exclu","Date de début","Date de fin","Règles mobilisées"]
rules_col = 'Règles mobilisées'
today = pd.Timestamp(datetime.utcnow().date()).strftime("%Y-%m-%d")
code_col = 'Code de la règle'
cols_drop = ['rule_norm', 'code_norm']
minister_col = "R_REL_MINIST"
statut = pd.read_excel(os.getenv("RGRH_STATUT_XLSX", "./FIP_STATUT_TYPPOP_25.00.00.xlsx"), engine="openpyxl")
population = pd.read_excel(os.getenv("RGRH_POPULATION_XLSX", "./POPULATION_25.00.00 (2).xlsx"), engine="openpyxl")


In [ ]:
# ...existing code...
# New cell: loop over all RGRH_*.xlsx (except statut/population) and compile results
from pathlib import Path
import pandas as pd

RGRH_DIR = Path(os.getenv("RGRH_DIR", "."))
EXCLUDE = {"FIP_STATUT_TYPPOP_25.00.00.xlsx", "POPULATION_25.00.00 (2).xlsx"}
out_frames = []

for f in sorted(RGRH_DIR.glob("RGRH_*.xlsx")):
    if f.name in EXCLUDE:
        continue
    try:
        print("Processing", f.name)
        xls = pd.ExcelFile(f, engine="openpyxl")
        dfs = {sheet: xls.parse(sheet) for sheet in xls.sheet_names}
        ref_axe_1 = dfs[xls.sheet_names[1]]
        ref_axe_2 = dfs[xls.sheet_names[4]]
        ref_axe_2['Date de fin']= pd.to_datetime(ref_axe_2['Date de fin'], errors='coerce')
        ref_axe_2=ref_axe_2[~ref_axe_2['Type de règle'].isin(['Contrôle'])]
        ref_axe_2=ref_axe_2[ref_axe_2['Date de fin'].isna() | (ref_axe_2['Date de fin'] >= today)]
        


        # filter contractuel (reuse cols_to_keep from earlier cell)
        contractuel = ref_axe_1.loc[
            ref_axe_1.get("Libellé Population", "").astype(str).str.lower().str.contains("contractuel", na=False)
        ].copy()
        existing = [c for c in cols_to_keep if c in contractuel.columns]
        contractuel = contractuel[existing]
        print(contractuel.shape)
        # apply same date / passant filter (uses `today` defined earlier)
        contractuel["Date de fin"] = pd.to_datetime(contractuel["Date de fin"], errors="coerce")
        contractuel["Passant / Exclu"] = contractuel["Passant / Exclu"].astype(str).str.lower()
        filtered_local = contractuel[
            contractuel["Passant / Exclu"].isin(["passant"])
            & (contractuel["Date de fin"].isna() | (contractuel["Date de fin"] >= today))
        ].copy()
        print(filtered_local.shape)
        # explode rules and normalize (reuses split_rules, norm_text, rules_col, code_col)
        df = filtered_local.copy()
        df[rules_col] = df[rules_col].fillna("").astype(str)
        df = df.assign(Règle=df[rules_col].apply(split_rules)).explode("Règle").reset_index(drop=True)
        df = df[df["Règle"].notna() & df["Règle"].astype(str).str.strip().astype(bool)].copy()
        df["rule_norm"] = df["Règle"].apply(norm_text).astype(str)

        ref = ref_axe_2.copy()
        ref[code_col] = ref[code_col].fillna("").astype(str)
        ref["code_norm"] = ref[code_col].apply(norm_text).astype(str)

        merged = df.merge(ref, left_on="rule_norm", right_on="code_norm", how="inner", suffixes=("", "_axe2"))
        print(merged.shape)
        contractuel_rules_expanded = merged.drop(columns=[c for c in ["rule_norm", "code_norm"] if c in merged.columns]).reset_index(drop=True)

        # merge population & statut (reuse variables `population` and `statut`)
        contractuel_rules_population = contractuel_rules_expanded.merge(
            population, left_on="Codification Population", right_on="Code population", how="left", suffixes=(None, "_pop")
        )
        contractuel_rules_population_statut = contractuel_rules_population.merge(
            statut, left_on="Code Statut", right_on="R_FOR_IDEN05", how="left", suffixes=(None, "_statut")
        )
        print(contractuel_rules_population_statut.shape)
        #filter for matte and interministériel
        minister_col = "R_REL_MINIST"
        s = contractuel_rules_population_statut[minister_col].fillna("").astype(str).str.strip()
        contractuel_rules_population_statut = contractuel_rules_population_statut[ s.str.upper().isin(["INTER", "MI190"])].copy()
        contractuel_rules_population_statut["source_file"] = f.name
        out_frames.append(contractuel_rules_population_statut)

    except Exception as e:
        print("Error processing", f.name, ":", e)

# concat and write single CSV
if out_frames:
    result = pd.concat(out_frames, ignore_index=True)
    result.to_csv("all_RGRH_donnees_personnelles_rules_population_statut.csv", index=False)
    print("Wrote", len(result), "rows to all_RGRH_donnees_personnelles_rules_population_statut.csv")
else:
    print("No RGRH files processed.")
#